# Mnemonics — CE Rerank Upgrade Eval (Kaggle, T4)

Üç CE modeli karşılaştırması, sonra en iyisini 500q'da doğrula.

**Kaggle ayarları:**
- Settings → Internet → **ON** (HF download için)
- Settings → Accelerator → **GPU T4 x2** (single T4 yeter ama x2 hız değil)
- Settings → Persistence → **Variables and Files** (sonuçlar save olsun)

Çalıştırma: Run All. Smoke 1dk, 100q ablation ~30-45dk (3 model), 500q v2-m3 ~50-70dk.

**Hedef:** MemPalace head-to-head R@1=0.920 / R@10=1.000.  
**Mevcut baseline (lokal, ms-marco-MiniLM, cand_k=50, augment_preferences):** R@1=0.846 / R@10=0.898.

## 1) Repo + bağımlılıklar

In [ ]:
!rm -rf /kaggle/working/mnemonics
!git clone https://github.com/nakata-app/mnemonics.git /kaggle/working/mnemonics
%cd /kaggle/working/mnemonics
!git log --oneline -5

In [ ]:
!pip install -q -e . sentence-transformers numpy adaptmem 2>&1 | tail -5
print('Install OK')

## 2) Dataset indir (HF → local JSON)

In [ ]:
import os
DATA = '/kaggle/working/longmemeval_s.json'
if not os.path.exists(DATA):
    print('Downloading dataset (~265 MB)...')
    !wget -q -O {DATA} \
        'https://huggingface.co/datasets/xiaowu0162/LongMemEval/resolve/main/longmemeval_s.json'
print(f'Size: {os.path.getsize(DATA)/1e6:.1f} MB')

In [ ]:
# Eval script DATA path'ini patch'le
import re, pathlib
p = pathlib.Path('/kaggle/working/mnemonics/benchmarks/longmemeval_eval.py')
src = p.read_text()
src = re.sub(r'DATA = Path\([^)]+\)', f'DATA = Path("{DATA}")', src)
p.write_text(src)
!grep -n 'DATA = Path' {p}

## 3) Smoke test — 5q, default reranker (ms-marco)

Pipeline'ın ayağa kalktığını doğrula.

In [ ]:
!cd /kaggle/working/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 5 --mode rerank \
    --augment-preferences --candidate-k 50 \
    --out /tmp/smoke.json && echo '=== SMOKE OK ==='

## 4) 100q ablation — 3 reranker × aynı sorular (apples-to-apples)

Aynı 100 soruyu üç CE modeliyle ardı ardına. Setup: `cand_k=50 --augment-preferences --seed=42` (production setup).

Tahmini süre (T4): ~10-15dk × 3 = 30-45dk.

In [ ]:
import os
os.makedirs('/kaggle/working/results', exist_ok=True)

# 4A) Baseline: ms-marco-MiniLM-L-12-v2 (varsayılan)
print('=== 4A: 100q — ms-marco-MiniLM-L-12-v2 (baseline) ===')
!cd /kaggle/working/mnemonics && \
    MNEMONICS_RERANK_MODEL=cross-encoder/ms-marco-MiniLM-L-12-v2 \
    python benchmarks/longmemeval_eval.py \
    --n 100 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /kaggle/working/results/lme100_msmarco.json \
    --per-q-out /kaggle/working/results/lme100_msmarco_perq.json

In [ ]:
# 4B) bge-reranker-base (orta güç, ~278M)
print('=== 4B: 100q — BAAI/bge-reranker-base ===')
!cd /kaggle/working/mnemonics && \
    MNEMONICS_RERANK_MODEL=BAAI/bge-reranker-base \
    python benchmarks/longmemeval_eval.py \
    --n 100 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /kaggle/working/results/lme100_bge_base.json \
    --per-q-out /kaggle/working/results/lme100_bge_base_perq.json

In [ ]:
# 4C) bge-reranker-v2-m3 (en güçlü, ~568M, multilingual)
print('=== 4C: 100q — BAAI/bge-reranker-v2-m3 ===')
!cd /kaggle/working/mnemonics && \
    MNEMONICS_RERANK_MODEL=BAAI/bge-reranker-v2-m3 \
    python benchmarks/longmemeval_eval.py \
    --n 100 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /kaggle/working/results/lme100_v2m3.json \
    --per-q-out /kaggle/working/results/lme100_v2m3_perq.json

In [ ]:
# 4D) Karşılaştırma tablosu
import json

results = {}
for name, path in [('ms-marco', 'lme100_msmarco'),
                   ('bge-base', 'lme100_bge_base'),
                   ('bge-v2-m3', 'lme100_v2m3')]:
    try:
        results[name] = json.load(open(f'/kaggle/working/results/{path}.json'))['mnemonics_rerank']
    except FileNotFoundError:
        results[name] = None

print(f'{"Reranker":15} {"R@1":>8} {"R@5":>8} {"R@10":>8} {"Runtime":>10}')
print('-' * 55)
for name, r in results.items():
    if r is None:
        print(f'{name:15} (missing)')
        continue
    print(f'{name:15} {r["R@1"]:>8.3f} {r["R@5"]:>8.3f} {r["R@10"]:>8.3f} {r["runtime_s"]:>9.0f}s')

print('\n=== KATEGORI BAZLI R@1 ===')
qtypes = sorted({qt for r in results.values() if r for qt in r['by_type']})
header = f'{"qtype":28} ' + ' '.join(f'{n:>10}' for n in results)
print(header)
print('-' * len(header))
for qt in qtypes:
    row = f'{qt:28} '
    for name in results:
        r = results[name]
        if r and qt in r['by_type']:
            row += f'{r["by_type"][qt]["R@1"]:>10.3f}'
        else:
            row += f'{"-":>10}'
    print(row)

## 5) 500q full eval — en iyi reranker

100q ablation'da en iyi R@1 veren modeli seç, 500q'da koştur. Mevcut baseline (lokal, 500q): R@1=0.846. Hedef: >0.90.

Tahmini süre: 50-90dk (model'e göre).

In [ ]:
# 100q sonuçlardan en iyiyi seç (en yüksek R@1)
best_name, best_model = None, None
model_map = {
    'ms-marco': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'bge-base': 'BAAI/bge-reranker-base',
    'bge-v2-m3': 'BAAI/bge-reranker-v2-m3',
}
if all(r is not None for r in results.values()):
    best_name = max(results, key=lambda n: results[n]['R@1'])
    best_model = model_map[best_name]
    print(f'En iyi 100q sonucu: {best_name} (R@1={results[best_name]["R@1"]:.3f})')
    print(f'500q için model: {best_model}')
else:
    print('100q sonuçları eksik, manuel seç:')
    print('  best_model = "BAAI/bge-reranker-v2-m3"  # ya da "BAAI/bge-reranker-base"')

In [ ]:
# 500q full sweep
assert best_model is not None, '100q sonuçları yok ya da model seçilmedi'
print(f'=== 5: 500q — {best_name} ({best_model}) ===')
!cd /kaggle/working/mnemonics && \
    MNEMONICS_RERANK_MODEL={best_model} \
    python benchmarks/longmemeval_eval.py \
    --n 500 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /kaggle/working/results/lme500_{best_name}.json \
    --per-q-out /kaggle/working/results/lme500_{best_name}_perq.json

In [ ]:
# 500q nihai özet
import json
r500 = json.load(open(f'/kaggle/working/results/lme500_{best_name}.json'))['mnemonics_rerank']
print(f'=== 500q FINAL ({best_name}) ===')
print(f'R@1={r500["R@1"]:.3f}  R@5={r500["R@5"]:.3f}  R@10={r500["R@10"]:.3f}  runtime={r500["runtime_s"]:.0f}s')
print()
print('Hedefler:')
print(f'  Lokal baseline (ms-marco): R@1=0.846, R@10=0.898')
print(f'  MemPalace head-to-head:    R@1=0.920, R@10=1.000')
print()
print('=== BY TYPE (R@1) ===')
for qt in sorted(r500['by_type']):
    b = r500['by_type'][qt]
    print(f'  {qt:28} n={b["n"]:3}  R@1={b["R@1"]:.3f}  R@10={b["R@10"]:.3f}')

## 6) Sonuç dosyaları

Kaggle session bitiminde sağ panelde **Output** sekmesinde `/kaggle/working/results/` altındaki tüm dosyalar download edilebilir.

In [ ]:
!ls -lh /kaggle/working/results/